- Issue: Wrong number of input channels for fno block, maybe correllated with lack of LP

In [7]:
import sys
import os

mt_path = os.path.abspath('/home/spacefi1/MT/MT-OpenSTL')
if mt_path not in sys.path:
    sys.path.append(mt_path)

from math import floor

import torch
import matplotlib.pyplot as plt
from openstl.models.fnolstm_model import FNOLSTM_B_Model
torch.manual_seed(1035)
from torch.utils.data import Dataset, DataLoader
from fvcore.nn import FlopCountAnalysis, flop_count_table
from icecream import install
install()
#
#ic.configureOutput(includeContext=True)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [2]:
class IdentityDataset(Dataset):
    def __init__(self, n):
        """
        Args:
            n (int): Number of samples in the dataset.
        """
        self.n = n  # Total number of samples
        self.data = torch.rand(n, 1, 8)  # Random data with shape (n_samples, channel=1, first_dimension=8)

    def __len__(self):
        # Return the total number of samples
        return self.n

    def __getitem__(self, idx):
        # The input and target are the same (identity function)
        sample = self.data[idx]
        return sample, sample  # Return input and target as identical
    

class DoublingDataset(Dataset):
    def __init__(self, n):
        """
        Args:
            n (int): Number of samples in the dataset.
            a = torch.tensor([[1,2,5],
                        [3,4,5]])
            b = torch.tensor([[2, 2]])
            torch.einsum('ij,ki->ij', a, b)
        """
        self.n = n  # Total number of 
        self.m = 8
        self.data = torch.ones(n, 1, self.m)
        mask = torch.arange(self.m) % 2 == 1
        self.data[:, :, mask] = self.data[:, :, mask] / 2

        self.labels = torch.cat((self.data, self.data), axis=-1)

        self.salt = torch.rand(n)[None, None, :]

        self.data = torch.einsum('ijk,lmi->ijk', self.data, self.salt)
        self.labels = torch.einsum('ijk,lmi->ijk', self.labels, self.salt)
        #self.data = []
        #for _ in range(10):
        #    new_data = torch.randn(n // 10, 1, round(torch.FloatTensor(1).uniform_(4, 16)[0].item()))
        #    ic(new_data.shape)
        #    self.data.append(new_data)
                    

    def __len__(self):
        # Return the total number of samples
        return self.n

    def __getitem__(self, idx):
        # The input and target are the same (identity function)
        
        sample = self.data[idx]
        label = self.labels[idx]
        return sample, label  # Return input and target as identical
    
class WavesDataset(Dataset):
    def __init__(self, n):
        """
        Args:
            n (int): Number of samples in the dataset.
        """
        self.n = n
        self.t, self.h, self.w = 4, 8, 8  # Total number of samples
        self.data = torch.ones(n, 1, self.t, self.h, self.w)
        mask = torch.arange(self.t) % 2 == 1
        self.data[:, :, mask, :, :] = self.data[:, :, mask, :, :] / 2

        #self.labels = torch.ones(n, 1, self.h, self.w)
        #mask = torch.arange(self.h) % 2 == 1
        #self.labels[:, :, mask, :] = self.labels[:, :, mask, :] / 2

        self.salt = torch.rand(n)[None, None, :, None, None]

        self.data = torch.einsum('abcde,mnaop->abcde', self.data, self.salt)
        self.coeff = torch.ones(n)[None, None, :, None, None] * 2
        self.labels = torch.einsum('abcde,mnaop->abcde', self.data, self.coeff)
        #self.labels = torch.einsum('ijkl,mnoi->ijkl', self.labels, self.salt)

    def __len__(self):
        # Return the total number of samples
        return self.n

    def __getitem__(self, idx):
        # The input and target are the same (identity function)
        sample = self.data[idx]
        label = self.labels[idx]
        return sample, label  # Return input and target as identical

In [20]:
num_hidden = 8
in_shape=(1,4,8,8)
fno_args = {
'model_type': 'skip',
'ndim': 2,    
'in_channels': num_hidden,
'out_channels': num_hidden,
'hidden_channels': 16,
'n_modes': 4,
'epochs': 41,
'step_size': 50,
'gamma': 0.5,
'lr': 1e-3,
'freq_print': 10}

fnolstm = FNOLSTM_B_Model(in_shape=in_shape, num_hidden=num_hidden, fno_block_args=fno_args) #FNO_Model('skip', n_modes, in_channels, out_channels, hidden_channels, ndim=2)
fnolstm.cuda()
print(f'Model params: {sum(p.numel() for p in fnolstm.parameters() if p.requires_grad)}')

batch = 32
train_size = 10_000
test_size = 200
train_set = WavesDataset(train_size)
test_set = WavesDataset(test_size)
#train_set = IdentityDataset(train_size)
#test_set = IdentityDataset(test_size)
train_loader = DataLoader(train_set, batch_size=batch, shuffle=True)
test_loader = DataLoader(test_set, batch_size=batch, shuffle=True)

Model params: 4680


In [22]:
C, T, H, W = in_shape
input_dummy = torch.ones(1, C, 8, H, W).to(device)
flops = FlopCountAnalysis(fnolstm, input_dummy)
flops = flop_count_table(flops)
#fps = measure_throughput(self.method.model.to(device), input_dummy)
#fps = 'Throughputs of {}: {:.3f}\n'.format(args.method, fps)

In [23]:
print(flops)

| module                              | #parameters or shape   | #flops    |
|:------------------------------------|:-----------------------|:----------|
| model                               | 4.68K                  | 1.641M    |
|  fnolstm                            |  4.672K                |  1.638M   |
|   fnolstm.FNO_block                 |   4.6K                 |   1.606M  |
|    fnolstm.FNO_block.lifting.nn     |    0.816K              |    0.344M |
|    fnolstm.FNO_block.fourier_layers |    2.976K              |    0.918M |
|    fnolstm.FNO_block.projection.nn  |    0.808K              |    0.344M |
|   fnolstm.LP                        |   72                   |   32.256K |
|    fnolstm.LP.weight                |    (8, 9, 1, 1)        |           |
|  h_to_out                           |  8                     |  3.584K   |
|   h_to_out.weight                   |   (1, 8, 1, 1)         |           |


In [4]:
a = next(iter(train_loader))


In [5]:
a[1].shape

torch.Size([32, 1, 4, 8, 8])

In [24]:
optimizer = torch.optim.AdamW(fnolstm.parameters(), lr=fno_args['lr'], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=fno_args['step_size'], gamma=fno_args['gamma'])
loss_func = torch.nn.MSELoss()

for epoch in range(fno_args['epochs']):
    train_mse = 0.0
    for step, (input, target) in enumerate(train_loader):
       input, target = input.cuda(), target.cuda()
       frames_tensor = torch.cat([input, target], dim=2)
       #ic(frames_tensor.shape)
       target = frames_tensor[:, :, 1:, :, :]
       optimizer.zero_grad()
       #ic(target.shape[2:])
       pred = fnolstm(frames_tensor)
       loss = loss_func(pred, target)
       loss.backward()
       optimizer.step()
       train_mse += loss.item()
    train_mse /= len(train_set)
    scheduler.step()

    with torch.no_grad():
        fnolstm.eval()
        test_relative_l2 = 0.0
        for step, (input, target) in enumerate(test_loader):
            input, target = input.cuda(), target.cuda()
            frames_tensor = torch.cat([input, target], dim=2)
            target = frames_tensor[:, :, 4:, :, :]
            pred = fnolstm(frames_tensor)[:, :, 3:, :, :]

            #ic(target.shape)
            #ic(pred.shape)
            loss = (torch.mean((pred - target) ** 2) / torch.mean(target ** 2)) ** 0.5 * 100
            test_relative_l2 += loss.item()
        test_relative_l2 /= len(test_set)
    
    if epoch % fno_args['freq_print'] == 0: print("######### Epoch:", epoch, " ######### Train Loss:", train_mse, " ######### Relative L2 Test Norm:", test_relative_l2)



    

######### Epoch: 0  ######### Train Loss: 0.005759578931331635  ######### Relative L2 Test Norm: 1.4254480171203614
######### Epoch: 10  ######### Train Loss: 2.2958229779760584e-06  ######### Relative L2 Test Norm: 0.026297835111618043
######### Epoch: 20  ######### Train Loss: 7.419590262998099e-07  ######### Relative L2 Test Norm: 0.04342716932296753
######### Epoch: 30  ######### Train Loss: 8.584004376871235e-07  ######### Relative L2 Test Norm: 0.04094577312469482
######### Epoch: 40  ######### Train Loss: 2.970382266312299e-07  ######### Relative L2 Test Norm: 0.013268440663814544


######### Epoch: 0  ######### Train Loss: 0.00640759562253952  ######### Relative L2 Test Norm: 1.5185894966125488
######### Epoch: 10  ######### Train Loss: 0.002194335534609854  ######### Relative L2 Test Norm: 1.2312910652160645
######### Epoch: 20  ######### Train Loss: 0.0020658159341663124  ######### Relative L2 Test Norm: 1.2052713203430176
######### Epoch: 30  ######### Train Loss: 0.0020534654323011636  ######### Relative L2 Test Norm: 1.2079949283599853
######### Epoch: 40  ######### Train Loss: 0.0020552436446771025  ######### Relative L2 Test Norm: 1.1935416221618653
Model params: 53569
torch.Size([32, 1, 4, 8, 8])